# Autonomous_Trailer
### ICS 3202 – Artificial Intelligence | Practical Exercise 4: Developing a Knowledge-Based Agent

**Project:** Self-driving trailer knowledge-based decision-support agent  
**Knowledge source:** Group Practical Exercise 3 – `PS-Agents/design.md`  
**Start:** Nairobi, Kenya | **Destination:** Kisumu, Kenya

**Group members:**
- 172651 – Abdilatif Ramadhan
- 191252 – Adam Etyang
- 189723 – Eleanor Linda Opolo

---

## 1. Introduction

This notebook implements a prototype knowledge-based agent for a self-driving trailer travelling through the Kenyan highway network. The agent stores road-network facts and operating-condition facts in a knowledge base. It applies explicit IF–THEN rules to assess a journey request and uses graph search to identify a minimum-distance route.



## 2. Imports and setup



In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Set
from collections import deque
import heapq
import json

## 3. Knowledge acquisition: road network from Practical Exercise 3

The following connections and distances are taken from the group's Exercise 3 `design.md`.

| Region | Towns represented |
|---|---|
| Nairobi | Nairobi |
| Central Highlands | Thika, Nyeri, Nyahururu |
| Rift Valley | Nakuru, Narok, Kilgoris, Kericho |
| Lake Victoria Basin | Kisii, Kisumu |



In [ ]:
ROAD_LINKS = [
    ("Nairobi", "Nakuru", 161),
    ("Nairobi", "Narok", 145),
    ("Nairobi", "Thika", 45),
    ("Thika", "Nyeri", 113),
    ("Nyeri", "Nyahururu", 104),
    ("Nyahururu", "Nakuru", 63),
    ("Narok", "Kilgoris", 176),
    ("Kilgoris", "Kisii", 56),
    ("Kisii", "Kisumu", 107),
    ("Nakuru", "Kericho", 107),
    ("Kericho", "Kisumu", 84),
]

REGIONS = {
    "Nairobi": ["Nairobi"],
    "Central Highlands": ["Thika", "Nyeri", "Nyahururu"],
    "Rift Valley": ["Nakuru", "Narok", "Kilgoris", "Kericho"],
    "Lake Victoria Basin": ["Kisii", "Kisumu"],
}

def build_graph(links):
    graph = {}
    for town_a, town_b, distance in links:
        if distance <= 0:
            raise ValueError("Road distances must be positive.")
        graph.setdefault(town_a, {})[town_b] = float(distance)
        graph.setdefault(town_b, {})[town_a] = float(distance)
    return graph

ROAD_GRAPH = build_graph(ROAD_LINKS)
TOWNS = sorted(ROAD_GRAPH)

print(f"Known towns: {len(TOWNS)}")
print(f"Two-way road connections: {len(ROAD_LINKS)}")
print("Towns:", ", ".join(TOWNS))

Known towns: 10
Two-way road connections: 11
Towns: Kericho, Kilgoris, Kisii, Kisumu, Nairobi, Nakuru, Narok, Nyahururu, Nyeri, Thika


## 4. Knowledge base representation

The knowledge base contains:
- **Structural facts:** towns, regions, and road connections with their recorded distances.
- **Dynamic/observed facts:** conditions supplied for a particular journey, such as road closures, weather, visibility and vehicle inspection results.
- **Rules:** logical conditions that infer an outcome from stored facts.



In [ ]:
@dataclass
class KnowledgeBase:
    graph: Dict[str, Dict[str, float]]
    regions: Dict[str, List[str]]
    facts: Dict[str, object] = field(default_factory=dict)
    sources: Dict[str, str] = field(default_factory=dict)

    def add_fact(self, name, value, source="User-provided journey observation"):
        self.facts[name] = value
        self.sources[name] = source

    def get_fact(self, name, default=None):
        return self.facts.get(name, default)

    def describe(self):
        return {
            "town_count": len(self.graph),
            "road_link_count": sum(len(v) for v in self.graph.values()) // 2,
            "regions": self.regions,
            "journey_facts": self.facts,
        }

kb = KnowledgeBase(graph=ROAD_GRAPH, regions=REGIONS)
print(json.dumps(kb.describe(), indent=2))

{
  "town_count": 10,
  "road_link_count": 11,
  "regions": {
    "Nairobi": [
      "Nairobi"
    ],
    "Central Highlands": [
      "Thika",
      "Nyeri",
      "Nyahururu"
    ],
    "Rift Valley": [
      "Nakuru",
      "Narok",
      "Kilgoris",
      "Kericho"
    ],
    "Lake Victoria Basin": [
      "Kisii",
      "Kisumu"
    ]
  },
  "journey_facts": {}
}


## 5. Knowledge representation: IF–THEN rules

The inference rules below model conservative decision support:

1. **Unknown location:** IF either requested town is not in the knowledge base, THEN request valid location information.
2. **Vehicle safety fault:** IF a critical vehicle inspection item fails, THEN do not recommend departure.
3. **Severe weather / poor visibility:** IF conditions are marked severe or visibility is below the configured threshold, THEN delay and request reassessment.
4. **Reported road closure:** IF a road segment is closed, THEN exclude that segment from route search.
5. **No remaining route:** IF all available connections are blocked or disconnected, THEN report that no route is available in the current knowledge base.
6. **Normal conditions:** IF the locations are known, the vehicle checks pass and no hold condition is reported, THEN provide a candidate shortest-distance route.



In [ ]:
VISIBILITY_HOLD_THRESHOLD_KM = 0.2  # Prototype parameter: 200 metres

def evaluate_conditions(observations):
    reasons = []
    status = "CLEAR"

    vehicle_checks = observations.get("vehicle_checks", {})
    failed_checks = [name for name, passed in vehicle_checks.items() if passed is False]
    if failed_checks:
        return {
            "decision": "DO_NOT_DEPART",
            "reasons": [f"Vehicle inspection failed: {', '.join(failed_checks)}."],
            "failed_checks": failed_checks,
        }

    if observations.get("severe_weather", False):
        status = "HOLD"
        reasons.append("Severe weather has been reported; delay the journey and reassess.")

    visibility = observations.get("visibility_km")
    if visibility is not None and visibility < VISIBILITY_HOLD_THRESHOLD_KM:
        status = "HOLD"
        reasons.append(
            f"Visibility ({visibility} km) is below the prototype threshold "
            f"({VISIBILITY_HOLD_THRESHOLD_KM} km)."
        )

    if observations.get("driver_fatigue", False):
        status = "HOLD"
        reasons.append("Driver fatigue has been reported; rest and reassess before travel.")

    if status == "HOLD":
        return {"decision": "DELAY_AND_REASSESS", "reasons": reasons, "failed_checks": []}

    if not vehicle_checks:
        reasons.append("No vehicle inspection results were supplied; human verification is required.")
        return {
            "decision": "HUMAN_REVIEW_REQUIRED",
            "reasons": reasons,
            "failed_checks": [],
        }

    if all(vehicle_checks.values()):
        reasons.append("All supplied vehicle inspection checks passed.")
    reasons.append("No configured hold condition was reported.")
    return {"decision": "ROUTE_PLANNING_ALLOWED", "reasons": reasons, "failed_checks": []}

## 6. Route search and path-cost calculation

The path-cost function is total distance in kilometres. Dijkstra's algorithm is used because all stored road distances are positive. Closed road segments are excluded during expansion.



In [ ]:
def shortest_route(graph, origin, destination, closed_roads=None):
    closed_roads = {frozenset(edge) for edge in (closed_roads or [])}

    if origin not in graph:
        raise ValueError(f"Unknown origin: {origin}")
    if destination not in graph:
        raise ValueError(f"Unknown destination: {destination}")

    frontier = [(0.0, origin, [origin])]
    best_cost = {origin: 0.0}

    while frontier:
        cost, town, path = heapq.heappop(frontier)

        if town == destination:
            return path, cost

        if cost > best_cost.get(town, float("inf")):
            continue

        for neighbour, distance in graph[town].items():
            if frozenset((town, neighbour)) in closed_roads:
                continue
            new_cost = cost + distance
            if new_cost < best_cost.get(neighbour, float("inf")):
                best_cost[neighbour] = new_cost
                heapq.heappush(frontier, (new_cost, neighbour, path + [neighbour]))

    return None, float("inf")


def route_distance(graph, path):
    return sum(graph[a][b] for a, b in zip(path, path[1:]))

route, distance = shortest_route(ROAD_GRAPH, "Nairobi", "Kisumu")
print("Shortest route:", " -> ".join(route))
print(f"Total distance: {distance:.0f} km")

Shortest route: Nairobi -> Nakuru -> Kericho -> Kisumu
Total distance: 352 km


## 7. Knowledge-based trailer agent

The agent combines location validation, condition assessment and route search. It returns both a decision and an explanation so that the user can see which facts affected the result.

The agent does not issue vehicle-control commands. A route recommendation is not a declaration that a road or vehicle is safe.

In [ ]:
def trailer_agent(
    knowledge_base,
    origin,
    destination,
    observations=None
):
    observations = observations or {}

    if origin not in knowledge_base.graph:
        return {
            "status": "NEEDS_INPUT",
            "decision": "UNKNOWN_ORIGIN",
            "explanation": [f"{origin!r} is not a known town in the knowledge base."],
            "route": None,
            "distance_km": None,
        }

    if destination not in knowledge_base.graph:
        return {
            "status": "NEEDS_INPUT",
            "decision": "UNKNOWN_DESTINATION",
            "explanation": [f"{destination!r} is not a known town in the knowledge base."],
            "route": None,
            "distance_km": None,
        }

    assessment = evaluate_conditions(observations)

    if assessment["decision"] != "ROUTE_PLANNING_ALLOWED":
        return {
            "status": "NOT_CLEARED",
            "decision": assessment["decision"],
            "explanation": assessment["reasons"],
            "route": None,
            "distance_km": None,
        }

    closed_roads = observations.get("closed_roads", [])
    path, cost = shortest_route(
        knowledge_base.graph, origin, destination, closed_roads
    )

    if path is None:
        return {
            "status": "NO_ROUTE",
            "decision": "HUMAN_REVIEW_REQUIRED",
            "explanation": [
                "No connected route remains after excluding reported closed roads."
            ],
            "route": None,
            "distance_km": None,
        }

    return {
        "status": "ROUTE_FOUND",
        "decision": "CANDIDATE_ROUTE",
        "explanation": assessment["reasons"] + [
            "The route is the minimum-distance route in the currently stored graph.",
            "Reported closed road segments were excluded from route search."
        ],
        "route": path,
        "distance_km": cost,
    }


def display_result(result):
    print("AGENT STATUS:", result["status"])
    print("DECISION:", result["decision"])
    for reason in result["explanation"]:
        print("-", reason)
    if result["route"]:
        print("ROUTE:", " -> ".join(result["route"]))
        print(f"DISTANCE: {result['distance_km']:.0f} km")

## 8. Demonstration 1: Nairobi to Kisumu under supplied normal conditions

For a route recommendation, the prototype requires explicit vehicle-check results. The sample below supplies a set of passed checks for demonstration.

In [ ]:
normal_observations = {
    "vehicle_checks": {
        "brakes": True,
        "tyres": True,
        "lights": True,
        "steering": True,
    },
    "severe_weather": False,
    "visibility_km": 5.0,
    "driver_fatigue": False,
    "closed_roads": [],
}

demo_result = trailer_agent(kb, "Nairobi", "Kisumu", normal_observations)
display_result(demo_result)

AGENT STATUS: ROUTE_FOUND
DECISION: CANDIDATE_ROUTE
- All supplied vehicle inspection checks passed.
- No configured hold condition was reported.
- The route is the minimum-distance route in the currently stored graph.
- Reported closed road segments were excluded from route search.
ROUTE: Nairobi -> Nakuru -> Kericho -> Kisumu
DISTANCE: 352 km


## 9. Demonstration 2: reported road closure

This scenario marks the Nakuru–Kericho connection as closed. The agent should exclude that edge and search for another connected route.

In [ ]:
closure_observations = {
    **normal_observations,
    "closed_roads": [("Nakuru", "Kericho")],
}

closure_result = trailer_agent(kb, "Nairobi", "Kisumu", closure_observations)
display_result(closure_result)

AGENT STATUS: ROUTE_FOUND
DECISION: CANDIDATE_ROUTE
- All supplied vehicle inspection checks passed.
- No configured hold condition was reported.
- The route is the minimum-distance route in the currently stored graph.
- Reported closed road segments were excluded from route search.
ROUTE: Nairobi -> Narok -> Kilgoris -> Kisii -> Kisumu
DISTANCE: 484 km


## 10. Demonstration 3: safety-related hold

This scenario reports a failed brake check. The rules should prevent route planning and return a do-not-depart decision.

In [ ]:
unsafe_observations = {
    **normal_observations,
    "vehicle_checks": {
        "brakes": False,
        "tyres": True,
        "lights": True,
        "steering": True,
    },
}

unsafe_result = trailer_agent(kb, "Nairobi", "Kisumu", unsafe_observations)
display_result(unsafe_result)

AGENT STATUS: NOT_CLEARED
DECISION: DO_NOT_DEPART
- Vehicle inspection failed: brakes.


## 11. Comparing the three routes recorded in Practical Exercise 3

Exercise 3 compared three candidate paths. The notebook calculates their totals from the graph so the arithmetic can be checked programmatically.

In [ ]:
CANDIDATE_ROUTES = {
    "Route A – via Nakuru and Kericho": [
        "Nairobi", "Nakuru", "Kericho", "Kisumu"
    ],
    "Route B – via Narok, Kilgoris and Kisii": [
        "Nairobi", "Narok", "Kilgoris", "Kisii", "Kisumu"
    ],
    "Route C – via Thika, Nyeri, Nyahururu, Nakuru and Kericho": [
        "Nairobi", "Thika", "Nyeri", "Nyahururu", "Nakuru", "Kericho", "Kisumu"
    ],
}

for name, path in CANDIDATE_ROUTES.items():
    total = route_distance(ROAD_GRAPH, path)
    print(f"{name}: {' -> '.join(path)} = {total:.0f} km")

best_name, best_path = min(
    CANDIDATE_ROUTES.items(),
    key=lambda item: route_distance(ROAD_GRAPH, item[1])
)
print("\nMinimum of the three Exercise 3 candidates:")
print(best_name, "=", route_distance(ROAD_GRAPH, best_path), "km")

Route A – via Nakuru and Kericho: Nairobi -> Nakuru -> Kericho -> Kisumu = 352 km
Route B – via Narok, Kilgoris and Kisii: Nairobi -> Narok -> Kilgoris -> Kisii -> Kisumu = 484 km
Route C – via Thika, Nyeri, Nyahururu, Nakuru and Kericho: Nairobi -> Thika -> Nyeri -> Nyahururu -> Nakuru -> Kericho -> Kisumu = 516 km

Minimum of the three Exercise 3 candidates:
Route A – via Nakuru and Kericho = 352.0 km


## 12. User interaction

The command-line interface accepts a start town and destination, plus a small set of condition inputs.

In [ ]:
def ask_yes_no(prompt):
    while True:
        answer = input(prompt + " (y/n): ").strip().lower()
        if answer in ("y", "yes"):
            return True
        if answer in ("n", "no"):
            return False
        print("Please enter y or n.")

def run_agent_cli():
    print("AUTONOMOUS TRAILER – KNOWLEDGE-BASED ROUTE ASSISTANT")
    print("Known towns:", ", ".join(TOWNS))
    origin = input("Origin: ").strip().title()
    destination = input("Destination: ").strip().title()

    checks = {
        "brakes": ask_yes_no("Did the brake check pass?"),
        "tyres": ask_yes_no("Did the tyre check pass?"),
        "lights": ask_yes_no("Did the lights check pass?"),
        "steering": ask_yes_no("Did the steering check pass?"),
    }
    severe_weather = ask_yes_no("Is severe weather reported?")
    fatigue = ask_yes_no("Is driver fatigue reported?")

    try:
        visibility = float(input("Visibility in km (e.g. 5): ").strip())
    except ValueError:
        visibility = None
        print("Visibility value unavailable; human review is recommended.")

    observations = {
        "vehicle_checks": checks,
        "severe_weather": severe_weather,
        "driver_fatigue": fatigue,
        "visibility_km": visibility,
        "closed_roads": [],
    }

    result = trailer_agent(kb, origin, destination, observations)
    display_result(result)



## 13. Testing and validation

The tests cover the road graph, route cost, unknown locations, safety rules, severe conditions and closure avoidance. They are simple software tests of the prototype's logic, not certification of a real autonomous vehicle.

In [ ]:
def run_tests():
    # Graph is undirected and distances match the Exercise 3 source data.
    assert ROAD_GRAPH["Nairobi"]["Nakuru"] == 161
    assert ROAD_GRAPH["Nakuru"]["Nairobi"] == 161
    assert len(ROAD_GRAPH) == 10

    # Known shortest route from Nairobi to Kisumu.
    path, cost = shortest_route(ROAD_GRAPH, "Nairobi", "Kisumu")
    assert path == ["Nairobi", "Nakuru", "Kericho", "Kisumu"]
    assert cost == 352

    # Unknown towns are rejected.
    unknown = trailer_agent(kb, "Nairobi", "Mombasa", normal_observations)
    assert unknown["decision"] == "UNKNOWN_DESTINATION"

    # Failed brake check prevents route recommendation.
    failed = trailer_agent(kb, "Nairobi", "Kisumu", unsafe_observations)
    assert failed["decision"] == "DO_NOT_DEPART"
    assert failed["route"] is None

    # Severe weather results in a delay.
    severe = trailer_agent(
        kb, "Nairobi", "Kisumu",
        {**normal_observations, "severe_weather": True}
    )
    assert severe["decision"] == "DELAY_AND_REASSESS"

    # A closed edge must not appear in the selected path.
    alternate = trailer_agent(kb, "Nairobi", "Kisumu", closure_observations)
    assert alternate["status"] == "ROUTE_FOUND"
    assert not any(
        frozenset((a, b)) == frozenset(("Nakuru", "Kericho"))
        for a, b in zip(alternate["route"], alternate["route"][1:])
    )

    print("All prototype tests passed.")

run_tests()

All prototype tests passed.


## 14. Limitations and possible improvements

The current prototype has the following limitations:

- **Static distances:** Road distances are copied from Exercise 3 and are not continuously updated.
- **Limited network:** Only the towns and road connections represented in the group's Exercise 3 graph are available.
- **No live traffic:** Congestion, accidents and travel-time estimates are not retrieved from a live service.
- **Manual condition inputs:** The prototype relies on supplied observations rather than physical sensors.
- **Simplified rules:** The rules cover a small set of example conditions and are not a complete vehicle-safety system.
- **No vehicle actuation:** The program recommends or withholds a candidate route; it does not control steering, braking or acceleration.

Possible improvements include integrating an authoritative road-closure/traffic source, recording timestamps and confidence for observations, expanding the road graph, adding a rule explanation trace, and validating safety rules with domain experts.